
# Cruzber — OOS Alert Ranking h=4 (BQML-aligned)
## Vertex AI Workbench · Colab Enterprise

**Pipeline:** h=4 v4 · KPI Steps 10 / 20 / 30

| Identidad | Propósito |
|-----------|-----------|
| **`hdeval@mda.isdi.es`** | Identidad única de la instancia Vertex AI — leer datos en `bucket-isdi-mda-online`, escribir outputs localmente |

**Data source (read-only):** `gs://bucket-isdi-mda-online/proyecto-troncal/cruzber/`  
**Output:** `outputs/{RUN_TS}/` — local al directorio del notebook en Vertex AI Workbench

| Step | Description |
|------|-------------|
| 10 | Enrich h4 forecast with unit economics (BQML formula: `p_oos × yhat_p50 × precio_unitario`) |
| 20 | Weekly Top-100 alert ranking by `risk_score` |
| 30 | Evaluate precision / recall / lift@100 vs Policy B baseline |

**Referencia BQML:** `thequantitativeledger.cruzber_models_eu.forecast_h4`  
**Benchmarks:** GLOBAL=11.08× · REST=13.12× · HIGH_SEASON=6.32×


## 1 · Instalar dependencias

In [1]:
# Ejecutar sólo la primera vez en el entorno Vertex AI Workbench
%pip install --quiet \
    google-cloud-storage>=2.14.0 \
    google-cloud-bigquery>=3.11.0 \
    db-dtypes>=1.1.0 \
    pandas>=2.0.0 \
    pyarrow>=12.0.0 \
    plotly>=5.18.0 \
    kaleido>=0.2.1

## 2 · Imports y autenticación

In [ ]:
import io, os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.cloud import storage
import google.auth

# ── Credenciales ADC de la instancia Vertex AI (hdeval@mda.isdi.es) ──────────
# La instancia Vertex AI corre íntegramente bajo hdeval@mda.isdi.es.
# google.auth.default() devuelve sus credenciales directamente desde
# el metadata server de la VM; no se necesita ninguna segunda cuenta.
credentials, project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
print(f"Identidad activa   : hdeval@mda.isdi.es")
print(f"Proyecto GCP       : {project}")
print(f"Tipo de credencial : {type(credentials).__name__}")

# Cliente GCS único — usa las credenciales de la instancia
gcs_data = storage.Client(credentials=credentials, project=project)
print(f"\nGCS client         : OK")
print("Listo para leer/escribir gs://bucket-isdi-mda-online/...")


Workbench project : thequantitativeledger
Workbench creds   : Credentials

  ⚠  ACCIÓN REQUERIDA
  La cuenta 'hdeval@mda.isdi.es' no está autenticada.
  Abre un terminal en este Workbench y ejecuta:
    gcloud auth login hdeval@mda.isdi.es
  Luego vuelve a ejecutar esta celda.



RuntimeError: Cuenta 'hdeval@mda.isdi.es' no autenticada en gcloud.

## 3 · Configuración del bucket y rutas

In [ ]:

import datetime

# ── Bucket y rutas ────────────────────────────────────────────────────────────
BUCKET_NAME  = "bucket-isdi-mda-online"
GCS_PREFIX   = "proyecto-troncal/cruzber"

# Archivos fuente disponibles en el bucket
SRC_FILES = {
    "fact_lineas_albaran" : "fact_lineas_albaran.csv",
    "dim_articulo"        : "dim_articulo.csv",
    "dim_familia"         : "dim_familia.csv",
    "dim_agrupacion"      : "dim_agrupacion_articulo.csv",
    "dim_fecha"           : "dim_fecha.csv",
}

# ── Parámetros del pipeline ───────────────────────────────────────────────────
H             = 4            # horizonte de forecast (semanas)
ROLL_W        = 12           # ventana de rolling histórico (semanas)
TOP_N         = 100          # alertas por semana en ranking
VAL_YEAR      = 2024         # año de validación (VAL split)

# IMPORTANTE: SEASON_MONTHS debe coincidir con los meses de mayor demanda real
# del catálogo en fact_lineas_albaran.
# → Si el catálogo es cycling: {4,5,6,7,8}  (temporada abril-agosto)
# → Si el catálogo es CRUZ accessories: revisar la distribución mensual
#   con la celda de diagnóstico (celda 11) y ajustar aquí.
# → Si no hay estacionalidad clara: usar todos los meses {1..12} para
#   evitar el artefacto p_oos=1.0 por panel de ceros estructurales.
SEASON_MONTHS = {4, 5, 6, 7, 8}   # ← AJUSTAR según diagnóstico si BQML ≠ cycling

# ── Run timestamp y prefijo de salida ────────────────────────────────────────
RUN_TS     = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
OUTPUT_PRE = f"{GCS_PREFIX}/outputs/{RUN_TS}"

print(f"Source  : gs://{BUCKET_NAME}/{GCS_PREFIX}/")
print(f"Output  : outputs/{RUN_TS}/ (local Vertex AI Workbench)")
print(f"Run TS  : {RUN_TS}")
print(f"Params  : H={H}  ROLL_W={ROLL_W}  TOP_N={TOP_N}  VAL_YEAR={VAL_YEAR}")
print(f"SEASON_MONTHS = {sorted(SEASON_MONTHS)}")

bucket = gcs_data.bucket(BUCKET_NAME)
blobs  = list(bucket.list_blobs(prefix=GCS_PREFIX, max_results=50))
print(f"\nArchivos en gs://{BUCKET_NAME}/{GCS_PREFIX}/  ({len(blobs)} objetos):")
for b in blobs:
    print(f"  {b.name}  ({b.size:,} bytes)")



## 4 · Carga de fuentes GCS y construcción de df_forecast / df_kpi

Los archivos disponibles son el **modelo dimensional crudo** de Cruzber. Este paso:
1. Carga `fact_lineas_albaran.csv`, `dim_articulo.csv` y tablas de dimensión desde GCS
2. **Benchmark de rentabilidad**: `precio_unit = base_imponible / cantidad` — base imposable neta (antes de IVA) dividida entre unidades entregadas, columna primaria de ingresos
3. Agrega ventas a nivel **SKU × semana**, restringido a **SEASON_EVAL = SEASON_MONTHS ∪ {mes previo}** = {3,4,5,6,7,8}. Excluir las semanas fuera de temporada evita que el rolling OOS interprete la demanda cero estructural de invierno como señal de rotura de stock.
4. Calcula features rolling h=4 sobre la serie de temporada: `p_oos_h4`, `q90_h4`, `yhat_p50_h4`
5. Construye `df_kpi` con precio unitario neto y margen estimados:
   - **Precio**: mediana de `base_imponible / cantidad` por SKU desde fact → fallback `precio_venta` en dim_articulo
   - **Coste**: `coste_escandallo` desde dim_articulo → fallback `precio_coste` desde fact


In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 4 · Carga de archivos fuente + construcción de df_forecast y df_kpi
#
# Alineado con el pipeline BQML cruzber_models_eu.forecast_h4:
#   - Columnas fuente: codigo_articulo, unidades, base_imponible
#   - Precio unitario: AVG(base_imponible / unidades) — igual que BQML
#   - Features: p_oos_h4, yhat_p50_h4, q90_h4, q95_h4, uncertainty_width
#   - Panel restringido a SEASON_EVAL para evitar artefacto p_oos=1 en off-season
# ══════════════════════════════════════════════════════════════════════════════

# ── Lector genérico CSV desde GCS ────────────────────────────────────────────
def read_gcs_csv(bucket_obj, prefix: str, filename: str, **kwargs) -> pd.DataFrame:
    data = bucket_obj.blob(f"{prefix}/{filename}").download_as_bytes()
    df   = pd.read_csv(io.BytesIO(data), low_memory=False, **kwargs)
    print(f"  ✓ {filename:<44}  {df.shape[0]:>8,} filas × {df.shape[1]:>3} cols")
    return df

print("── Cargando archivos fuente desde GCS…\n")
fact   = read_gcs_csv(bucket, GCS_PREFIX, SRC_FILES["fact_lineas_albaran"])
dim_a  = read_gcs_csv(bucket, GCS_PREFIX, SRC_FILES["dim_articulo"])
dim_f  = read_gcs_csv(bucket, GCS_PREFIX, SRC_FILES["dim_familia"])
dim_ag = read_gcs_csv(bucket, GCS_PREFIX, SRC_FILES["dim_agrupacion"])
dim_d  = read_gcs_csv(bucket, GCS_PREFIX, SRC_FILES["dim_fecha"])

print("\n── Columnas detectadas:")
for name, df in [("fact_lineas_albaran", fact), ("dim_articulo", dim_a), ("dim_fecha", dim_d)]:
    print(f"  {name}: {list(df.columns)}")

# ── Detección flexible de columnas ───────────────────────────────────────────
# Orden de prioridad = nombres reales en BQML primero, aliases de fallback después
_ALIASES = {
    "sku_id"      : ["codigo_articulo","articulo_id","sku_id","sku","cod_articulo","id_articulo"],
    "fecha"       : ["fecha_albaran","fecha","fecha_entrega","fecha_factura","fec_albaran","fec_entrega"],
    # BQML usa 'unidades' como columna de volumen
    "uds"         : ["unidades","cantidad","uds_servidas","qty","cantidad_entregada","uds_pedidas"],
    # BQML usa 'base_imponible' = base imposable neta (antes IVA) = benchmark de ingresos
    "importe_neto": ["base_imponible","importe_neto","total_neto","importe_venta_neto",
                     "venta_neta","importe","total_importe"],
    # Precio de tarifa en dim_articulo
    "precio_neto" : ["precio_venta","precio_neto","precio_unit_neto","pvp_neto",
                     "precio_venta_neto","precio_unitario_neto","importe_unitario","precio"],
    "coste_unit"  : ["coste_escandallo","coste_unit","coste_unitario","costo_unit",
                     "precio_coste","precio_compra","coste_medio"],
}

def _col(df, key):
    """Devuelve el nombre real de la columna canónica en df (búsqueda exacta y luego parcial)."""
    for alias in _ALIASES.get(key, [key]):
        if alias in df.columns:
            return alias
    key_low = key.lower()
    for col in df.columns:
        if key_low in col.lower():
            return col
    raise KeyError(f"Columna '{key}' no encontrada. Disponibles: {list(df.columns)}")

# ══════════════════════════════════════════════════════════════════════════════
# 4A · Normalizar fact_lineas_albaran
# Benchmark BQML: precio_unitario = AVG(base_imponible / unidades)
# ══════════════════════════════════════════════════════════════════════════════
f = fact.copy()
f["sku_id"]        = f[_col(fact, "sku_id")].astype(str).str.strip()
f["fecha"]         = pd.to_datetime(f[_col(fact, "fecha")], dayfirst=True, errors="coerce")
f["unidades"]      = pd.to_numeric(f[_col(fact, "uds")], errors="coerce").fillna(0).clip(lower=0)
rev_col            = _col(fact, "importe_neto")      # resuelve base_imponible si existe
f["base_imponible"]= pd.to_numeric(f[rev_col], errors="coerce").fillna(0)
print(f"  ✓ Columna volumen  : '{_col(fact, 'uds')}'")
print(f"  ✓ Columna ingresos : '{rev_col}'  (median line={f['base_imponible'].median():.2f})")

f = f.dropna(subset=["fecha"])
f["week"] = f["fecha"].dt.to_period("W-MON").apply(lambda p: p.start_time)

# ══════════════════════════════════════════════════════════════════════════════
# 4B · Panel semanal SKU × semana (restringido a SEASON_EVAL)
#
# Un SKU de ciclismo tiene demanda = 0 en invierno por estacionalidad estructural,
# NO por rotura. El rolling OOS sobre el año completo produce p_oos = 1.0 como
# artefacto. Restringimos a SEASON_EVAL (temporada + pre-stocking) para que el
# rolling interprete correctamente las semanas de cero como OOS real.
# ══════════════════════════════════════════════════════════════════════════════
PRE_SEASON  = {(m - 2) % 12 + 1 for m in SEASON_MONTHS}   # {3} si temporada empieza en Abril
SEASON_EVAL = SEASON_MONTHS | PRE_SEASON                   # {3,4,5,6,7,8}
print(f"\n  Meses de evaluación (SEASON_EVAL): {sorted(SEASON_EVAL)}")

weekly_s = (
    f[f["week"].dt.month.isin(SEASON_EVAL)]
     .groupby(["sku_id", "week"], sort=True)
     .agg(y_true=("unidades", "sum"), base_imponible=("base_imponible", "sum"))
     .reset_index()
)

# Ciclo de vida activo por SKU dentro de la temporada
sku_range_s = weekly_s.groupby("sku_id")["week"].agg(first_w="min", last_w="max")

def _expand_season_panel(grp):
    sku = grp.name
    if sku not in sku_range_s.index:
        return grp
    all_w   = pd.date_range(sku_range_s.loc[sku, "first_w"],
                            sku_range_s.loc[sku, "last_w"], freq="W-MON")
    sw = all_w[all_w.month.isin(SEASON_EVAL)]
    if len(sw) == 0:
        return grp
    return (
        grp.set_index("week")[["y_true", "base_imponible"]]
           .reindex(sw, fill_value=0)
           .assign(sku_id=sku)
           .rename_axis("week")
           .reset_index()
    )

print(f"  Expandiendo panel de temporada por ciclo de vida de SKU…")
panel = (
    weekly_s.groupby("sku_id", group_keys=False)
            .apply(_expand_season_panel)
            .reset_index(drop=True)
)
print(f"  SKUs únicos  : {panel['sku_id'].nunique():,}")
print(f"  Filas panel  : {panel.shape[0]:,}  "
      f"({panel['week'].min().date()} → {panel['week'].max().date()})")

# ══════════════════════════════════════════════════════════════════════════════
# 4C · Rolling features h=4 — alineado con forecast_h4 de BQML
#
# Columnas producidas (matching cruzber_models_eu.forecast_h4):
#   p_oos_h4          → probabilidad de OOS en [t+1, t+H]
#   yhat_p50_h4       → demanda esperada (mediana rolling)
#   q90_h4, q95_h4    → cuantiles de demanda esperada
#   uncertainty_width → q95 − q90 (amplitud intervalo de incertidumbre)
#   y_true_h4         → demanda real H semanas adelante (ground truth)
#
# min_periods = MAX(4, ROLL_W//3): permite calcular features en SKUs con
# historial corto de temporada. Con min_periods=ROLL_W, SKUs con < ROLL_W
# semanas históricas se dropnan → solo quedan los muy OOS → p_oos = 1.0.
# ══════════════════════════════════════════════════════════════════════════════
MIN_P = max(4, ROLL_W // 3)   # 4 para ROLL_W=12; escala proporcionalmente
print(f"\n  Calculando rolling features (roll={ROLL_W}w, min_periods={MIN_P}, h={H}w)…")
panel = panel.sort_values(["sku_id", "week"])

def _add_features(grp):
    y  = grp["y_true"]
    r  = y.rolling(ROLL_W, min_periods=MIN_P)
    g  = grp.copy()
    g["p_oos_h4"]    = (y == 0).rolling(ROLL_W, min_periods=MIN_P).mean().shift(1)
    g["yhat_p50_h4"] = r.median().shift(1)
    g["q90_h4"]      = r.quantile(0.90).shift(1)
    g["q95_h4"]      = r.quantile(0.95).shift(1)
    g["y_true_h4"]   = y.shift(-H)
    return g

panel = panel.groupby("sku_id", group_keys=False).apply(_add_features)
panel = panel.dropna(subset=["p_oos_h4", "y_true_h4"]).reset_index(drop=True)

# uncertainty_width = q95 − q90  (matching BQML alerts_top100_h4.uncertainty_width_p95)
panel["uncertainty_width"] = (panel["q95_h4"] - panel["q90_h4"]).clip(lower=0)

panel["target_week"]       = panel["week"] + pd.Timedelta(weeks=H)
panel["stockout_event_h4"] = (panel["y_true_h4"] == 0).astype("Int64")
panel["season_group"]      = panel["week"].dt.month.map(
    lambda m: "HIGH_SEASON" if m in SEASON_MONTHS else "REST"
)
panel["split"] = panel["week"].dt.year.map(
    lambda yr: "VAL" if yr >= VAL_YEAR else "TRAIN"
)
panel.rename(columns={"week": "decision_week"}, inplace=True)

df_forecast = panel[[
    "sku_id", "split", "decision_week", "target_week",
    "p_oos_h4", "yhat_p50_h4", "q90_h4", "q95_h4", "uncertainty_width",
    "y_true_h4", "stockout_event_h4", "season_group"
]].copy()

print(f"  df_forecast: {df_forecast.shape[0]:,} filas  |  splits: "
      f"{df_forecast['split'].value_counts().to_dict()}")
print(f"  p_oos_h4   —  media={df_forecast['p_oos_h4'].mean():.3f}  "
      f"std={df_forecast['p_oos_h4'].std():.3f}  "
      f"(esperado: media<0.9, std>0)")
print(f"  q90_h4     —  median={df_forecast['q90_h4'].median():.1f}")
print(f"  q95_h4     —  median={df_forecast['q95_h4'].median():.1f}")

# ══════════════════════════════════════════════════════════════════════════════
# 4D · precio_unitario desde fact — AVG(base_imponible / unidades)
#      Idéntico a la CTE precios_sku de consultas_informe_h4.sql:
#      AVG(SAFE_DIVIDE(base_imponible, NULLIF(unidades, 0)))
# ══════════════════════════════════════════════════════════════════════════════
kpi = dim_a.copy()
sku_col_dim = _col(dim_a, "sku_id")
kpi.rename(columns={sku_col_dim: "sku_id"}, inplace=True)
kpi["sku_id"] = kpi["sku_id"].astype(str).str.strip()

# Precio unitario = promedio de (base_imponible / unidades) por SKU — igual que BQML
ventas_val = f[(f["unidades"] > 0) & (f["base_imponible"] > 0)].copy()
ventas_val["pu"] = ventas_val["base_imponible"] / ventas_val["unidades"]
ventas_val = ventas_val[ventas_val["pu"] > 0]   # SAFE_DIVIDE / NULLIF equivalent

precio_por_sku = (
    ventas_val.groupby("sku_id")["pu"]
              .mean()                           # AVG — matching BQML
              .rename("precio_unitario_fact")
              .reset_index()
)
kpi = kpi.merge(precio_por_sku, on="sku_id", how="left")
print(f"\n  precio_unitario (AVG base_imponible/unidades): "
      f"median={precio_por_sku['precio_unitario_fact'].median():.2f}€  "
      f"cobertura={len(precio_por_sku):,}/{len(kpi):,} SKUs")

# Precio de tarifa desde dim_articulo como fallback
try:
    pnet_col = _col(dim_a, "precio_neto")
    kpi["precio_unitario"] = kpi["precio_unitario_fact"].fillna(
        pd.to_numeric(kpi[pnet_col], errors="coerce").replace(0, np.nan)
    )
    print(f"  ✓ precio_unitario: fact AVG + fallback dim_a['{pnet_col}']")
except KeyError:
    kpi["precio_unitario"] = kpi["precio_unitario_fact"]
    print("  ⚠ Solo precio desde fact.")

# Coste unitario: coste_escandallo (para margen_pct informativo, no usado en ranking)
try:
    cost_col = _col(dim_a, "coste_unit")
    kpi["coste_unitario"] = pd.to_numeric(kpi[cost_col], errors="coerce").replace(0, np.nan)
    print(f"  ✓ coste_unitario  : '{cost_col}'  median={kpi['coste_unitario'].median():.2f}€")
except KeyError:
    kpi["coste_unitario"] = np.nan
    print("  ⚠ coste_unitario no encontrado.")

# margen_pct informativo (no usado en el ranking de riesgo BQML)
kpi["margen_unitario"] = kpi["precio_unitario"].fillna(0) - kpi["coste_unitario"].fillna(0)
kpi["margen_pct"] = np.where(
    kpi["precio_unitario"].fillna(0) > 0,
    kpi["margen_unitario"] / kpi["precio_unitario"], np.nan
).round(4)

# Fechas de vida del SKU
lifecycle = (
    f.groupby("sku_id")
     .agg(primera_venta=("fecha", "min"), ultima_venta=("fecha", "max"))
     .reset_index()
)
kpi = kpi.merge(lifecycle, on="sku_id", how="left")
kpi["dias_en_catalogo"] = (kpi["ultima_venta"] - kpi["primera_venta"]).dt.days

last_date   = f["fecha"].max()
active_skus = f[f["fecha"] >= last_date - pd.Timedelta(days=90)]["sku_id"].unique()
kpi["sku_active"] = kpi["sku_id"].isin(active_skus).astype(int)

# Renombrar columnas a nombres canónicos
_renames = {}
for canon, aliases in {
    "descripcion_articulo": ["descripcion_articulo","descripcion","nombre_articulo","denominacion","nom_articulo"],
    "codigo_familia"       : ["codigo_familia","cod_familia","familia","id_familia"],
    "codigo_subfamilia"    : ["codigo_subfamilia","cod_subfamilia","subfamilia","id_subfamilia"],
    "tipo_abc"             : ["tipo_abc","abc","clasificacion_abc","clase_abc"],
    "estado_articulo"      : ["estado_articulo","estado","activo","estado_sku"],
    "obsoleto"             : ["obsoleto","baja","descatalogado","es_obsoleto"],
    "area_competencia_lc"  : ["area_competencia_lc","area_competencia","area","linea_competencia"],
}.items():
    for al in aliases:
        if al in kpi.columns and al != canon:
            _renames[al] = canon
            break
kpi.rename(columns=_renames, inplace=True)

KPI_KEEP = [
    "sku_id", "descripcion_articulo", "codigo_familia", "codigo_subfamilia",
    "area_competencia_lc", "tipo_abc", "estado_articulo", "obsoleto",
    "sku_active", "primera_venta", "ultima_venta", "dias_en_catalogo",
    "precio_unitario", "coste_unitario", "margen_unitario", "margen_pct"
]
df_kpi = (kpi[[c for c in KPI_KEEP if c in kpi.columns]]
          .drop_duplicates(subset=["sku_id"])
          .reset_index(drop=True))

print(f"\n  df_kpi : {df_kpi.shape[0]:,} SKUs  |  cols: {list(df_kpi.columns)}")
print(f"  SKUs activos (90d): {df_kpi['sku_active'].sum():,}")
print(f"  precio_unitario   — nulos: {df_kpi['precio_unitario'].isna().sum():,}  "
      f"median={df_kpi['precio_unitario'].median():.2f}")


## 4E · `df_v_kpi` — Snapshot `v_kpi_por_articulo` en Pandas

Replica la vista BigQuery `v_kpi_por_articulo` íntegramente en Pandas, a partir de
`fact_lineas_albaran` y `dim_articulo` ya cargados en memoria.

Columnas producidas (spec completa):

| Grupo | Columnas |
|---|---|
| **Actividad** | `lineas_articulo`, `clientes_articulo`, `albaranes_articulo` |
| **Volumen/valor** | `unidades_articulo`, `base_imponible_articulo`, `importe_coste_articulo` |
| **Rentabilidad** | `margen_articulo`, `margen_porcentual_articulo` |
| **Ciclo de vida** | `primera_venta`, `ultima_venta`, `dias_en_catalogo` |
| **Métricas/unidad** | `precio_unit_net = base_imponible / uds`, `margen_unit = margen / uds` (≥ 0) |

`margen_unit` se usa en la celda siguiente para calcular `eur_at_risk_margin = p_oos_h4 × q90_h4 × margen_unit`.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4E · df_v_kpi — replica de la vista v_kpi_por_articulo en Pandas
#
# Fuentes: fact (raw DataFrame) + f (fact normalizado) + dim_a
# No requiere BigQuery. Usa únicamente DataFrames ya en memoria.
# ══════════════════════════════════════════════════════════════════════════════
from datetime import datetime as _dt
import pathlib as _pl

# ── 1. Helpers para columnas opcionales ──────────────────────────────────────
def _opt(df, key):
    """Devuelve nombre de columna en df o None si no existe."""
    try:
        return _col(df, key)
    except KeyError:
        return None

# Columnas opcionales en fact
_cli_col  = _opt(fact, "codigo_cliente")      # → clientes_articulo
_alb_col  = _opt(fact, "codigo_albaran")      # → albaranes_articulo
_COSTE_ALIASES = [
    "importe_coste", "coste_total", "total_coste", "importe_compra",
    "coste_mercancia", "coste_linea", "total_coste_linea",
]
_cost_col = next((c for c in _COSTE_ALIASES if c in fact.columns), None)

print(f"[v_kpi] columna cliente  : {_cli_col or '— no detectada'}")
print(f"[v_kpi] columna albaran  : {_alb_col or '— no detectada'}")
print(f"[v_kpi] columna coste    : {_cost_col or '— no detectada (se usará coste_escandallo×uds)'}")

# ── 2. DataFrame de trabajo: sku_id normalizado + columnas numéricas ──────────
_fk = fact.copy()
_fk["sku_id"] = _fk[_col(fact, "sku_id")].astype(str).str.strip()
_fk["_uds"]   = pd.to_numeric(_fk[_col(fact, "uds")],         errors="coerce").fillna(0).clip(lower=0)
_fk["_rev"]   = pd.to_numeric(_fk[_col(fact, "importe_neto")], errors="coerce").fillna(0)
_fk["_cost"]  = (
    pd.to_numeric(_fk[_cost_col], errors="coerce").fillna(0)
    if _cost_col
    else pd.Series(np.nan, index=_fk.index)
)

# ── 3. Fallback coste: coste_escandallo × unidades (cuando no hay columna coste en fact) ──
if _cost_col is None:
    try:
        _ce_col = _col(dim_a, "coste_unit")   # coste_escandallo
        _ce_map = (
            dim_a[[_col(dim_a, "sku_id"), _ce_col]]
            .copy()
            .rename(columns={_col(dim_a, "sku_id"): "sku_id", _ce_col: "_ce"})
            .assign(sku_id=lambda d: d["sku_id"].astype(str).str.strip())
            .drop_duplicates("sku_id")
        )
        _fk = _fk.merge(_ce_map, on="sku_id", how="left")
        _fk["_cost"] = (_fk["_ce"].fillna(0) * _fk["_uds"]).clip(lower=0)
        print(f"[v_kpi] coste estimado via coste_escandallo × unidades")
    except KeyError:
        print("[v_kpi] ⚠ coste no disponible — importe_coste_articulo = NaN")

# ── 4. Agregaciones por SKU ───────────────────────────────────────────────────
_kpi_agg = (
    _fk.groupby("sku_id", sort=False)
       .agg(
           unidades_articulo        = ("_uds",  "sum"),
           base_imponible_articulo  = ("_rev",  "sum"),
           importe_coste_articulo   = ("_cost", "sum"),
       )
       .reset_index()
)

# Clientes únicos (si existe)
if _cli_col:
    _kpi_agg = _kpi_agg.merge(
        _fk.groupby("sku_id")[_cli_col].nunique().rename("clientes_articulo").reset_index(),
        on="sku_id", how="left",
    )
else:
    _kpi_agg["clientes_articulo"] = np.nan

# Albaranes únicos (si existe)
if _alb_col:
    _kpi_agg = _kpi_agg.merge(
        _fk.groupby("sku_id")[_alb_col].nunique().rename("albaranes_articulo").reset_index(),
        on="sku_id", how="left",
    )
else:
    _kpi_agg["albaranes_articulo"] = np.nan

# ── 5. Ciclo de vida + lineas_articulo (ya disponible en f normalizado) ───────
_lifecycle = (
    f.groupby("sku_id")
     .agg(
         primera_venta   = ("fecha", "min"),
         ultima_venta    = ("fecha", "max"),
         lineas_articulo = ("fecha", "size"),
     )
     .reset_index()
)
_kpi_agg = _kpi_agg.merge(_lifecycle, on="sku_id", how="left")
_kpi_agg["dias_en_catalogo"] = (_kpi_agg["ultima_venta"] - _kpi_agg["primera_venta"]).dt.days

# ── 6. Márgenes totales y por unidad ─────────────────────────────────────────
_kpi_agg["margen_articulo"] = (
    _kpi_agg["base_imponible_articulo"] - _kpi_agg["importe_coste_articulo"].fillna(0)
)
_kpi_agg["margen_porcentual_articulo"] = np.where(
    _kpi_agg["base_imponible_articulo"] > 0,
    100 * _kpi_agg["margen_articulo"] / _kpi_agg["base_imponible_articulo"],
    np.nan,
).round(4)

_safe_uds = _kpi_agg["unidades_articulo"].replace(0, np.nan)
_kpi_agg["precio_unit_net"] = (_kpi_agg["base_imponible_articulo"] / _safe_uds).round(4)
_kpi_agg["margen_unit"]     = (_kpi_agg["margen_articulo"]         / _safe_uds).clip(lower=0).round(4)

# ── 7. Merge con dim_articulo para columnas descriptivas ─────────────────────
_DIM_RENAME = {
    "descripcion_articulo" : ["descripcion_articulo","descripcion","nombre_articulo","denominacion"],
    "codigo_familia"        : ["codigo_familia","cod_familia","familia"],
    "codigo_subfamilia"     : ["codigo_subfamilia","cod_subfamilia","subfamilia"],
    "descripcion_subfamilia": ["descripcion_subfamilia","desc_subfamilia","nombre_subfamilia"],
    "agrupacion_listado"    : ["agrupacion_listado","agrupacion","cod_agrupacion"],
    "descripcion_agrupacion": ["descripcion_agrupacion","desc_agrupacion","nombre_agrupacion"],
    "sub_agrupacion_listado": ["sub_agrupacion_listado","sub_agrupacion","cod_sub_agrupacion"],
    "area_competencia_lc"   : ["area_competencia_lc","area_competencia","area"],
    "estado_articulo"       : ["estado_articulo","estado","activo"],
    "tipo_abc"              : ["tipo_abc","abc","clasificacion_abc"],
    "obsoleto"              : ["obsoleto","baja","descatalogado"],
}
_dim_slim = dim_a.copy()
_skucol   = _col(dim_a, "sku_id")
_dim_slim.rename(columns={_skucol: "sku_id"}, inplace=True)
_dim_slim["sku_id"] = _dim_slim["sku_id"].astype(str).str.strip()

_dim_r = {}
for canon, aliases in _DIM_RENAME.items():
    for al in aliases:
        if al in _dim_slim.columns and al != canon:
            _dim_r[al] = canon
            break
_dim_slim.rename(columns=_dim_r, inplace=True)
_dim_keep = ["sku_id"] + [c for c in _DIM_RENAME if c in _dim_slim.columns]
_dim_slim = _dim_slim[_dim_keep].drop_duplicates("sku_id")

_kpi_agg = _kpi_agg.merge(_dim_slim, on="sku_id", how="left")
_kpi_agg["snapshot_ts"] = _dt.utcnow().isoformat()

# ── 8. Ordenar columnas según spec v_kpi_por_articulo ────────────────────────
_V_KPI_ORDER = [
    "sku_id", "descripcion_articulo", "codigo_familia", "codigo_subfamilia",
    "descripcion_subfamilia", "agrupacion_listado", "descripcion_agrupacion",
    "sub_agrupacion_listado", "area_competencia_lc", "estado_articulo",
    "tipo_abc", "obsoleto",
    "lineas_articulo", "clientes_articulo", "albaranes_articulo",
    "unidades_articulo", "base_imponible_articulo", "importe_coste_articulo",
    "margen_articulo", "margen_porcentual_articulo",
    "primera_venta", "ultima_venta", "dias_en_catalogo",
    "precio_unit_net", "margen_unit",
    "snapshot_ts",
]
df_v_kpi = _kpi_agg[[c for c in _V_KPI_ORDER if c in _kpi_agg.columns]].copy()

# ── 9. Snapshot a disco ───────────────────────────────────────────────────────
_snap_dir = _pl.Path(os.getcwd()) / "outputs"
_snap_dir.mkdir(parents=True, exist_ok=True)
df_v_kpi.to_parquet(_snap_dir / "kpi_por_articulo_snapshot.parquet", index=False)
df_v_kpi.to_csv(    _snap_dir / "kpi_por_articulo_snapshot.csv",     index=False)

# ── 10. Resumen ───────────────────────────────────────────────────────────────
print(f"\ndf_v_kpi : {df_v_kpi.shape[0]:,} SKUs  ×  {df_v_kpi.shape[1]} columnas")
print(f"Snapshot : outputs/kpi_por_articulo_snapshot.parquet  (+ .csv)\n")
_cols_num = [c for c in ["unidades_articulo","base_imponible_articulo",
                          "margen_articulo","margen_porcentual_articulo",
                          "precio_unit_net","margen_unit"] if c in df_v_kpi.columns]
print(df_v_kpi[_cols_num].describe().round(2).to_string())


## 5 · Exploración y saneamiento de datos

In [ ]:

# ── df_forecast ──────────────────────────────────────────────────────────────
print("=== FORECAST ===")
print(df_forecast.dtypes)
print(f"\nSplits       : {df_forecast['split'].value_counts().to_dict()}")
print(f"Season groups: {df_forecast['season_group'].value_counts().to_dict()}")
print(f"\nDistribución p_oos_h4:")
print(df_forecast["p_oos_h4"].describe().round(4))
print(f"\nNulos por columna clave:")
for col in ["p_oos_h4", "q90_h4", "q95_h4", "yhat_p50_h4", "y_true_h4", "stockout_event_h4"]:
    n = df_forecast[col].isna().sum()
    print(f"  {col}: {n:,}")

# ══════════════════════════════════════════════════════════════════════════════
# GUARD: Verificar que SEASON_EVAL contiene semanas con demanda positiva
# Si yhat_p50_h4 es siempre 0 → SEASON_MONTHS no corresponde con la temporada real
# ══════════════════════════════════════════════════════════════════════════════
pos_yhat = (df_forecast["yhat_p50_h4"] > 0).sum()
pos_ytrue = (df_forecast["y_true_h4"] > 0).sum()
pct_oos_1 = (df_forecast["p_oos_h4"] == 1.0).mean() * 100

print(f"\n{'='*60}")
print(f"DIAGNÓSTICO SEASON_EVAL (meses={sorted(SEASON_EVAL)})")
print(f"  yhat_p50_h4 > 0  : {pos_yhat:,} filas ({pos_yhat/len(df_forecast)*100:.1f}%)")
print(f"  y_true_h4   > 0  : {pos_ytrue:,} filas ({pos_ytrue/len(df_forecast)*100:.1f}%)")
print(f"  p_oos_h4 == 1.0  : {pct_oos_1:.1f}% de las filas")

if pos_yhat == 0 or pct_oos_1 > 95:
    print()
    print("⚠  ALERTA: panel estacional con demanda=0 en todos los SKUs.")
    print("   → SEASON_MONTHS no coincide con los meses de venta reales del catálogo.")
    print("   Distribución de ventas mensuales reales en fact_lineas_albaran:")
    monthly = (
        f[f["unidades"] > 0]
         .assign(mes=f["fecha"].dt.month)
         .groupby("mes")["unidades"]
         .agg(n_transacciones="count", uds_total="sum")
         .rename_axis("mes")
         .reset_index()
    )
    print(monthly.to_string(index=False))
    top_months = monthly.nlargest(5, "uds_total")["mes"].tolist()
    print(f"\n   Top-5 meses por volumen: {sorted(top_months)}")
    print(f"   → Ajusta SEASON_MONTHS en la celda de configuración a: {set(top_months)}")
else:
    print("  ✓ Panel estacional con demanda positiva — SEASON_MONTHS correcto.")
print(f"{'='*60}")

# ── Diagnóstico de escala: fact_lineas_albaran ────────────────────────────────
print("\n=== DIAGNÓSTICO ESCALA — fact_lineas_albaran ===")
KEYWORDS_FACT = ["base_imponible","imponible","importe","precio","neto","pvp","coste","uds","cantidad"]
print("Columnas de volumen/importe disponibles:")
for col in fact.columns:
    if any(k in col.lower() for k in KEYWORDS_FACT):
        s = pd.to_numeric(fact[col], errors="coerce")
        print(f"  {col:<35}  min={s.min():.4f}  median={s.median():.4f}  max={s.max():.4f}  nulos={s.isna().sum()}")

# Benchmark precio_unit
print(f"\nBenchmark precio_unit = base_imponible / unidades (ventas positivas con base>0):")
_rev_col = _col(fact, "importe_neto")
_uds_col = _col(fact, "uds")
_mask = (pd.to_numeric(fact[_uds_col], errors="coerce") > 0) & \
        (pd.to_numeric(fact[_rev_col], errors="coerce") > 0)
_sample = fact[_mask].copy()
_sample["precio_unit_est"] = (pd.to_numeric(_sample[_rev_col], errors="coerce") /
                               pd.to_numeric(_sample[_uds_col], errors="coerce"))
print(f"  Columna de ingresos usada: '{_rev_col}'")
print(_sample["precio_unit_est"].describe().round(4))
print(f"  → Valores negativos : {(_sample['precio_unit_est'] < 0).sum():,}")

# ── Diagnóstico de escala: dim_articulo ──────────────────────────────────────
print("\n=== DIAGNÓSTICO ESCALA — dim_articulo ===")
for col in dim_a.columns:
    if any(k in col.lower() for k in ["coste","precio","neto","pvp","escandallo","margen","compra","venta"]):
        s = pd.to_numeric(dim_a[col], errors="coerce")
        print(f"  {col:<35}  min={s.min():.4f}  median={s.median():.4f}  max={s.max():.4f}  nulos={s.isna().sum()}")

# ── df_kpi ────────────────────────────────────────────────────────────────────
print("\n=== KPI (post-construcción) ===")
print(df_kpi.dtypes)
print(f"\nSKUs únicos         : {df_kpi['sku_id'].nunique():,}")
print(f"SKUs activos (90d)  : {df_kpi['sku_active'].sum():,}")
for col in ["precio_unitario", "coste_unitario", "margen_unitario", "margen_pct"]:
    if col in df_kpi.columns:
        s = df_kpi[col].astype(float)
        print(f"\n  {col}:")
        print(f"    min={s.min():.4f}  median={s.median():.4f}  max={s.max():.4f}  nulos={s.isna().sum()}")
        print(f"    negativos={(s < 0).sum():,}  ceros={(s == 0).sum():,}")
    else:
        print(f"  {col}: COLUMNA NO DISPONIBLE")

if "tipo_abc" in df_kpi.columns:
    print(f"\nDistribución tipo_abc:")
    print(df_kpi["tipo_abc"].value_counts().head(10))



## 6 · Step 10 — Enriquecer forecast con economía unitaria

Replica la CTE `impacto_high_season` de `consultas_informe_h4.sql` (pipeline BQML).  
Fórmulas centrales — idénticas a BQML:

$$\text{venta\_esperada\_eur} = \hat{y}_{p50,h4} \times \text{precio\_unitario}$$

$$\text{riesgo\_stockout\_eur} = p\_oos_{h4} \times \hat{y}_{p50,h4} \times \text{precio\_unitario}$$

$$\text{riesgo\_q95\_eur} = p\_oos_{h4} \times q95_{h4} \times \text{precio\_unitario}$$

`risk_score = riesgo_stockout_eur` — campo de ranking en `alerts_top100_h4`.  
`uncertainty_width = q95_{h4} - q90_{h4}` — amplitud del intervalo de incertidumbre.  
`precio_unitario = AVG(base\_imponible / unidades)` — idéntico a la CTE `precios_sku` en BQML.


In [ ]:

# ── Columnas KPI a transferir al forecast enriquecido ─────────────────────────
KPI_COLS = [
    "sku_id",
    "descripcion_articulo", "codigo_familia", "codigo_subfamilia",
    "area_competencia_lc", "tipo_abc",
    "estado_articulo", "obsoleto", "sku_active",
    "primera_venta", "ultima_venta", "dias_en_catalogo",
    "precio_unitario", "coste_unitario", "margen_unitario", "margen_pct",
]
kpi_slim = df_kpi[[c for c in KPI_COLS if c in df_kpi.columns]].copy()

# ── JOIN forecast × kpi ───────────────────────────────────────────────────────
df_enriched = df_forecast.merge(kpi_slim, on="sku_id", how="left")

# ══════════════════════════════════════════════════════════════════════════════
# Fórmulas económicas — alineadas con consultas_informe_h4.sql (BQML)
#
# BQML (CTE impacto_high_season):
#   venta_esperada_eur   = yhat_p50_h4 × precio_unitario
#   riesgo_stockout_eur  = p_oos_h4 × yhat_p50_h4 × precio_unitario
#   riesgo_q95_eur       = p_oos_h4 × q95_h4     × precio_unitario
#   risk_score           = riesgo_stockout_eur   (campo de ranking en alerts)
#
# La fórmula usa precio_unitario (ingresos), no margen.
# El margen_pct es informativo para análisis posterior.
# ══════════════════════════════════════════════════════════════════════════════
pu = df_enriched["precio_unitario"].fillna(0)

df_enriched["venta_esperada_eur"]  = df_enriched["yhat_p50_h4"] * pu
df_enriched["riesgo_stockout_eur"] = df_enriched["p_oos_h4"] * df_enriched["yhat_p50_h4"] * pu
df_enriched["riesgo_q95_eur"]      = df_enriched["p_oos_h4"] * df_enriched["q95_h4"]      * pu

# risk_score = riesgo_stockout_eur (matching alerts_top100_h4.risk_score en BQML)
df_enriched["risk_score"] = df_enriched["riesgo_stockout_eur"]

# Flag de match KPI
df_enriched["kpi_matched"] = df_enriched["precio_unitario"].notna().astype(int)

# Estadísticas de match
n_total   = len(df_enriched)
n_match   = df_enriched["kpi_matched"].sum()
n_val     = df_enriched[df_enriched["split"] == "VAL"]["sku_id"].nunique()
match_pct = 100 * n_match / n_total

print(f"Filas enriquecidas   : {n_total:,}")
print(f"KPI match            : {n_match:,}  ({match_pct:.1f}%)")
print(f"SKUs únicos en VAL   : {n_val:,}")

val_mask = df_enriched["split"] == "VAL"
print(f"\nEstadísticas económicas (VAL):")
for col in ["venta_esperada_eur", "riesgo_stockout_eur", "riesgo_q95_eur"]:
    s = df_enriched.loc[val_mask, col]
    print(f"  {col:<25}  sum={s.sum():>12,.0f}  median={s.median():>8.2f}")

# ══════════════════════════════════════════════════════════════════════════════
# Margen por unidad desde df_v_kpi (v_kpi_por_articulo snapshot)
#   precio_unit_net = base_imponible_articulo / unidades_articulo   (AVG de ventas totales)
#   margen_unit     = margen_articulo / unidades_articulo            (≥ 0, clamped)
#   eur_at_risk_margin = p_oos_h4 × q90_h4 × margen_unit
# ══════════════════════════════════════════════════════════════════════════════
_vkpi_merge = df_v_kpi[["sku_id", "precio_unit_net", "margen_unit"]].copy()
df_enriched = df_enriched.merge(_vkpi_merge, on="sku_id", how="left")

_mu = df_enriched["margen_unit"].fillna(0)
df_enriched["eur_at_risk_margin"] = df_enriched["p_oos_h4"] * df_enriched["q90_h4"] * _mu

_n_mu = (df_enriched["margen_unit"] > 0).sum()
print(f"\nMargen match (margen_unit > 0) : {_n_mu:,} / {n_total:,}")
s_mar = df_enriched.loc[val_mask, "eur_at_risk_margin"]
print(f"eur_at_risk_margin (VAL)       : sum={s_mar.sum():>12,.2f}  median={s_mar.median():>8.4f}")



## 7 · Step 20 — Ranking semanal Top-100 (alineado con `alerts_top100_h4`)

Replica la lógica de ordenación de `cruzber_models_eu.alerts_top100_h4`.  
Criterio de ordenación por semana de decisión:

1. `sku_active = 1` primero  (activo en los últimos 90 días)
2. `risk_score DESC`  (`= riesgo_stockout_eur = p_oos_h4 × yhat_p50_h4 × precio_unitario`)
3. `p_oos_h4 DESC`   (desempate 1)
4. `uncertainty_width DESC`  (`= q95_h4 − q90_h4`, desempate 2)

Label de verdad terrena: `true_stockout_label_model` (matching `alerts_top100_h4`).


In [ ]:

# Restricción al split VAL (evaluación)
val = df_enriched[df_enriched["split"] == "VAL"].copy()

# ── Labels de verdad-terrena (matching BQML alerts_top100_h4) ────────────────
# true_stockout_label_model = stockout en H semanas según el modelo (y_true_h4 == 0)
val["true_stockout_label_model"] = val["stockout_event_h4"].fillna(0).astype(int)
# true_stockout_sales0 = semana sin ventas en target_week (definición alternativa)
val["true_stockout_sales0"]      = (val["y_true_h4"].fillna(0) == 0).astype(int)

# ── Criterio de ranking — alineado con BQML alerts_top100_h4 ─────────────────
# Ordenación por semana:
#   1. sku_active = 1 primero  (SKUs activos en los últimos 90 días)
#   2. risk_score DESC          (= riesgo_stockout_eur = p_oos × yhat_p50 × precio)
#   3. p_oos_h4 DESC            (desempate 1)
#   4. uncertainty_width DESC   (desempate 2: mayor incertidumbre = más urgente)
val["_active_sort"] = val["sku_active"].fillna(0).map(lambda x: 0 if x == 1 else 1)

val_ranked = (
    val
    .sort_values(
        ["decision_week", "_active_sort", "risk_score", "p_oos_h4", "uncertainty_width"],
        ascending=[True, True, False, False, False]
    )
    .assign(rank_in_week=lambda df: df.groupby("decision_week").cumcount() + 1)
)

alerts_margin = val_ranked[val_ranked["rank_in_week"] <= TOP_N].copy()
alerts_margin.drop(columns=["_active_sort"], inplace=True)

# ── Margin-ranked alerts: Top-100 por eur_at_risk_margin ─────────────────────
# Criterio alternativo basado en margen en riesgo:
#   1. sku_active = 1 primero
#   2. eur_at_risk_margin DESC  (= p_oos_h4 × q90_h4 × margen_unit)
#   3. p_oos_h4 DESC            (desempate 1)
#   4. uncertainty_width DESC   (desempate 2)
val_mr = (
    val
    .sort_values(
        ["decision_week", "_active_sort", "eur_at_risk_margin", "p_oos_h4", "uncertainty_width"],
        ascending=[True, True, False, False, False]
    )
    .assign(rank_margin=lambda df: df.groupby("decision_week").cumcount() + 1)
)
alerts_top100_h4_margin = val_mr[val_mr["rank_margin"] <= TOP_N].copy()
alerts_top100_h4_margin.drop(columns=["_active_sort", "rank_margin"], inplace=True, errors="ignore")

# ── Resumen comparativo ───────────────────────────────────────────────────────
n_weeks  = alerts_margin["decision_week"].nunique()
n_alerts = len(alerts_margin)
print(f"Semanas de decisión cubiertas : {n_weeks}")
print(f"Alertas totales (Top-{TOP_N}) revenue-ranked  : {n_alerts:,}")
print(f"Alertas totales (Top-{TOP_N}) margin-ranked   : {len(alerts_top100_h4_margin):,}")

print(f"\nTop-5 alertas más críticas por eur_at_risk_margin:")
_margin_cols = [c for c in [
    "decision_week", "sku_id", "descripcion_articulo",
    "p_oos_h4", "q90_h4", "margen_unit",
    "eur_at_risk_margin", "riesgo_stockout_eur",
    "true_stockout_label_model"
] if c in alerts_top100_h4_margin.columns]
print(
    alerts_top100_h4_margin[_margin_cols]
    .nlargest(5, "eur_at_risk_margin")
    .to_string(index=False)
)


## 8 · Step 30 — Evaluación: precision / recall / lift@100

Replica `kpi/30_eval_alerts_top100_h4_margin_pooled.sql`.

$$\text{lift@100} = \frac{\text{precision@100}}{\text{prevalencia base}}$$

Referencia pipeline Cloud Run: **GLOBAL=11.08×  |  REST=13.12×  |  HIGH\_SEASON=6.32×**

In [ ]:

def evaluate_alerts(alerts_df: pd.DataFrame, universe_df: pd.DataFrame,
                    label_col: str = "true_stockout_label_model") -> pd.DataFrame:
    """
    Calcula precision@100, recall@100, lift@100 y agregados económicos.
    Replica cruzber_models_eu.run_summary_h4 métricas prec100/rec100/lift100.
    """
    rows = []
    groups = list(universe_df["season_group"].unique()) + ["ALL"]

    for sg in groups:
        if sg == "ALL":
            u = universe_df
            a = alerts_df
        else:
            u = universe_df[universe_df["season_group"] == sg]
            a = alerts_df[alerts_df["season_group"] == sg]

        n_universe  = len(u)
        n_stockouts = u[label_col].sum()
        prevalence  = n_stockouts / n_universe if n_universe else 0

        n_alerts    = len(a)
        n_tp        = a[label_col].sum()
        precision   = n_tp / n_alerts   if n_alerts   else 0
        recall      = n_tp / n_stockouts if n_stockouts else 0
        lift        = precision / prevalence if prevalence else 0

        # Métricas económicas (matching consultas_informe_h4.sql query 4 & 5)
        sum_riesgo = a["riesgo_stockout_eur"].fillna(0).sum()
        sum_venta  = a["venta_esperada_eur"].fillna(0).sum()
        sum_q95    = a["riesgo_q95_eur"].fillna(0).sum()
        avg_riesgo = a["riesgo_stockout_eur"].fillna(0).mean()
        avg_width  = a["uncertainty_width"].fillna(0).mean()

        rows.append({
            "season_group"           : sg,
            "n_universe"             : n_universe,
            "n_stockouts"            : int(n_stockouts),
            "prevalence"             : round(prevalence, 4),
            "n_alerts"               : n_alerts,
            "n_true_positives"       : int(n_tp),
            "precision_at_100"       : round(precision, 4),
            "recall_at_100"          : round(recall, 4),
            "lift_at_100"            : round(lift, 2),
            "sum_venta_esperada_eur" : round(sum_venta, 2),
            "sum_riesgo_stockout_eur": round(sum_riesgo, 2),
            "sum_riesgo_q95_eur"     : round(sum_q95, 2),
            "avg_riesgo_top100"      : round(avg_riesgo, 4),
            "avg_uncertainty_width"  : round(avg_width, 4),
        })

    return pd.DataFrame(rows)


# ── Evaluar con las dos definiciones de label (matching BQML) ─────────────────
eval_model  = evaluate_alerts(alerts_margin, val, label_col="true_stockout_label_model")
eval_sales0 = evaluate_alerts(alerts_margin, val, label_col="true_stockout_sales0")

eval_final = eval_model.merge(
    eval_sales0[["season_group","precision_at_100","recall_at_100","lift_at_100",
                 "n_true_positives","n_stockouts"]].rename(columns={
                     "precision_at_100" : "precision_at_100_sales0",
                     "recall_at_100"    : "recall_at_100_sales0",
                     "lift_at_100"      : "lift_at_100_sales0",
                     "n_true_positives" : "n_tp_sales0",
                     "n_stockouts"      : "n_stockouts_sales0",
                 }),
    on="season_group", how="left",
)

print("=== EVALUACIÓN ALERTAS TOP-100 (matching cruzber_models_eu.run_summary_h4) ===\n")
print(eval_final[[
    "season_group", "n_universe", "prevalence",
    "precision_at_100", "recall_at_100", "lift_at_100",
    "sum_riesgo_stockout_eur", "avg_riesgo_top100"
]].to_string(index=False))

print("\nReferencia BQML (run_summary_h4):")
print("  HIGH_SEASON: prec100=?  rec100=?  lift100=?  (ver run_summary_h4)")
print("  Benchmarks anteriores: GLOBAL=11.08×  REST=13.12×  HIGH_SEASON=6.32×")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Step 30b · Evaluación de alertas margin-ranked (eur_at_risk_margin)
#
# Replica eval_alerts_top100_h4_margin_pooled:
#   - precision@100, recall@100, lift@100  por season_group y pooled
#   - sum/avg eur_at_risk_margin en Top-100
#   - Comparativa revenue-rank (risk_score) vs margin-rank (eur_at_risk_margin)
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_alerts_margin(alerts_df: pd.DataFrame, universe_df: pd.DataFrame,
                            label_col: str = "true_stockout_label_model") -> pd.DataFrame:
    """
    Extiende evaluate_alerts() con métricas de margen:
      sum/avg eur_at_risk_margin, avg margen_unit en el Top-100.
    """
    rows = []
    groups = list(universe_df["season_group"].unique()) + ["ALL"]

    for sg in groups:
        u = universe_df if sg == "ALL" else universe_df[universe_df["season_group"] == sg]
        a = alerts_df   if sg == "ALL" else alerts_df[alerts_df["season_group"]   == sg]

        n_universe  = len(u)
        n_stockouts = u[label_col].sum()
        prevalence  = n_stockouts / n_universe if n_universe else 0

        n_alerts  = len(a)
        n_tp      = a[label_col].sum()
        precision = n_tp / n_alerts    if n_alerts    else 0
        recall    = n_tp / n_stockouts if n_stockouts else 0
        lift      = precision / prevalence if prevalence else 0

        # Métricas de margen
        sum_eur_margin = a["eur_at_risk_margin"].fillna(0).sum()
        avg_eur_margin = a["eur_at_risk_margin"].fillna(0).mean()
        avg_mu         = a["margen_unit"].fillna(0).mean()
        sum_riesgo_rev = a["riesgo_stockout_eur"].fillna(0).sum()   # para comparativa

        rows.append({
            "season_group"                    : sg,
            "n_universe"                      : n_universe,
            "n_stockouts"                     : int(n_stockouts),
            "prevalence"                      : round(prevalence, 4),
            "n_alerts"                        : n_alerts,
            "n_true_positives"                : int(n_tp),
            "precision_at_100"                : round(precision, 4),
            "recall_at_100"                   : round(recall, 4),
            "lift_at_100"                     : round(lift, 2),
            "sum_eur_at_risk_margin_top100"   : round(sum_eur_margin, 2),
            "avg_eur_at_risk_margin_top100"   : round(avg_eur_margin, 4),
            "avg_margen_unit_top100"          : round(avg_mu, 4),
            "sum_riesgo_stockout_eur_top100"  : round(sum_riesgo_rev, 2),
        })

    return pd.DataFrame(rows)


eval_margin_alerts = evaluate_alerts_margin(
    alerts_top100_h4_margin, val, label_col="true_stockout_label_model"
)

# ── Tabla comparativa: revenue-rank vs margin-rank ────────────────────────────
_rev_eval = evaluate_alerts(alerts_margin, val, label_col="true_stockout_label_model")

_comp_margin = (
    _rev_eval[["season_group", "lift_at_100", "sum_riesgo_stockout_eur"]]
    .rename(columns={"lift_at_100": "lift_revenue", "sum_riesgo_stockout_eur": "riesgo_rev_eur"})
    .merge(
        eval_margin_alerts[["season_group", "lift_at_100",
                             "sum_eur_at_risk_margin_top100", "avg_margen_unit_top100"]]
        .rename(columns={"lift_at_100": "lift_margin"}),
        on="season_group",
    )
)
_comp_margin["delta_lift"] = (_comp_margin["lift_margin"] - _comp_margin["lift_revenue"]).round(2)

print("=== EVALUACIÓN MARGIN-RANKED Top-100 (eur_at_risk_margin) ===\n")
print(eval_margin_alerts[[
    "season_group", "n_universe", "prevalence",
    "precision_at_100", "recall_at_100", "lift_at_100",
    "sum_eur_at_risk_margin_top100", "avg_eur_at_risk_margin_top100",
]].to_string(index=False))

print("\n=== Comparativa Revenue-Rank vs Margin-Rank ===\n")
print(_comp_margin.to_string(index=False))


In [ ]:

# ── Comparación: ranking BQML (risk_score = p_oos×yhat_p50×precio)
#                vs ranking estándar Policy B (p_oos × q90_h4) ────────────────
val["score_policy_b"] = val["p_oos_h4"] * val["q90_h4"]

val_std_ranked = (
    val
    .sort_values(["decision_week", "score_policy_b"], ascending=[True, False])
    .assign(rank_std=lambda df: df.groupby("decision_week").cumcount() + 1)
)
alerts_std = val_std_ranked[val_std_ranked["rank_std"] <= TOP_N].copy()
eval_std   = evaluate_alerts(alerts_std, val, label_col="true_stockout_label_model")

# Tabla comparativa
comp = eval_model[["season_group","lift_at_100","sum_riesgo_stockout_eur"]].rename(
    columns={"lift_at_100":"lift_bqml","sum_riesgo_stockout_eur":"riesgo_bqml"}
).merge(
    eval_std[["season_group","lift_at_100","sum_riesgo_stockout_eur"]].rename(
        columns={"lift_at_100":"lift_std","sum_riesgo_stockout_eur":"riesgo_std"}
    ),
    on="season_group"
)
comp["delta_lift"]   = (comp["lift_bqml"]  - comp["lift_std"]).round(2)
comp["delta_riesgo"] = (comp["riesgo_bqml"] - comp["riesgo_std"]).round(2)
comp["riesgo_mejora_%"] = (
    (comp["delta_riesgo"] / comp["riesgo_std"].replace(0, np.nan)) * 100
).round(1)

print("=== Ranking BQML (risk_score) vs Ranking Estándar (Policy B: p_oos×q90) ===\n")
print(comp.to_string(index=False))


## 9 · Visualizaciones

In [ ]:

# ── Fig 1: Lift@100 — Ranking BQML vs Policy B por temporada ─────────────────
fig1 = go.Figure()

fig1.add_bar(
    x=comp["season_group"], y=comp["lift_bqml"],
    name="Ranking BQML (risk_score)", marker_color="#1f77b4"
)
fig1.add_bar(
    x=comp["season_group"], y=comp["lift_std"],
    name="Ranking Estándar (Policy B: p_oos×q90)", marker_color="#aec7e8"
)
fig1.update_layout(
    title="Lift@100 — BQML risk_score vs Policy B por temporada",
    yaxis_title="Lift@100",
    barmode="group",
    template="plotly_white",
    height=420,
    legend=dict(orientation="h", yanchor="bottom", y=1.02)
)
fig1.show()


In [ ]:

# ── Fig 2: Riesgo stockout acumulado por semana (perfil estacional) ───────────
weekly_riesgo = (
    alerts_margin
    .groupby(["decision_week","season_group"], sort=True)
    .agg(
        sum_riesgo_stockout_eur=("riesgo_stockout_eur", "sum"),
        sum_riesgo_q95_eur=("riesgo_q95_eur", "sum"),
        n_alerts=("sku_id", "count")
    )
    .reset_index()
)

fig2 = px.bar(
    weekly_riesgo,
    x="decision_week", y="sum_riesgo_stockout_eur",
    color="season_group",
    color_discrete_map={"HIGH_SEASON": "#e74c3c", "REST": "#3498db"},
    title="Riesgo de Rotura (p_oos × yhat_p50 × precio) — Top-100 alerts, VAL 2024",
    labels={"sum_riesgo_stockout_eur": "€ riesgo stockout", "decision_week": "Semana de decisión"},
    template="plotly_white",
    height=420
)
fig2.update_xaxes(dtick="M1", tickformat="%b %Y")
fig2.show()


In [ ]:

# ── Fig 3: Scatter p_oos vs riesgo_stockout_eur (coloreado por tipo_abc) ─────
sample = alerts_margin.dropna(subset=["riesgo_stockout_eur","p_oos_h4"]).sample(
    min(3000, len(alerts_margin)), random_state=42
)

fig3 = px.scatter(
    sample,
    x="p_oos_h4", y="riesgo_stockout_eur",
    color="tipo_abc",
    size="yhat_p50_h4",
    size_max=18,
    hover_data=["sku_id","descripcion_articulo","precio_unitario","season_group","uncertainty_width"],
    opacity=0.65,
    title="P(OOS) vs Riesgo de Rotura (€) — Top-100 alerts, muestra VAL 2024",
    labels={"p_oos_h4": "P(OOS) h=4", "riesgo_stockout_eur": "€ riesgo stockout"},
    template="plotly_white",
    height=500
)
fig3.add_vline(x=0.5, line_dash="dot", line_color="gray",
               annotation_text="umbral p=0.50")
fig3.show()


In [ ]:

# ── Fig 4: Top-20 SKUs por riesgo acumulado (matching query 7 del informe H4) ─
top20 = (
    alerts_margin
    .groupby(["sku_id","descripcion_articulo","tipo_abc","codigo_familia"], sort=False)
    .agg(
        total_riesgo_stockout_eur=("riesgo_stockout_eur", "sum"),
        total_riesgo_q95_eur=("riesgo_q95_eur", "sum"),
        n_semanas_en_alerta=("decision_week", "nunique"),
        avg_p_oos=("p_oos_h4", "mean"),
        avg_precio_unitario=("precio_unitario", "mean"),
        avg_yhat_p50=("yhat_p50_h4", "mean"),
        n_stockouts_reales=("true_stockout_label_model", "sum"),
    )
    .reset_index()
    .nlargest(20, "total_riesgo_stockout_eur")
)

fig4 = px.bar(
    top20.sort_values("total_riesgo_stockout_eur"),
    x="total_riesgo_stockout_eur",
    y=top20.sort_values("total_riesgo_stockout_eur")["sku_id"].astype(str)
      + " – " + top20.sort_values("total_riesgo_stockout_eur")["descripcion_articulo"].fillna(""),
    color="tipo_abc",
    orientation="h",
    title="Top-20 SKUs por Riesgo Stockout Acumulado — VAL 2024 (matching BQML query 7)",
    labels={"x": "€ riesgo acumulado", "y": "SKU"},
    template="plotly_white",
    height=560
)
fig4.update_layout(showlegend=True, yaxis_title="")
fig4.show()

print("\nTabla Top-20 (matching cruzber_models_eu.forecast_h4 TOP 10 query):")
print(top20[[
    "sku_id", "descripcion_articulo",
    "total_riesgo_stockout_eur", "total_riesgo_q95_eur",
    "n_semanas_en_alerta", "avg_p_oos", "avg_precio_unitario", "n_stockouts_reales"
]].to_string(index=False))


## 10 · Exportar resultados

Los artefactos se guardan localmente en `outputs/{RUN_TS}/` dentro del directorio del notebook (`os.getcwd()`), que en Vertex AI Workbench coincide con la carpeta donde reside el fichero `.ipynb`.


In [ ]:

import pathlib
from datetime import datetime

NOTEBOOK_DIR = pathlib.Path(os.getcwd())
RUN_TS       = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
OUT_DIR      = NOTEBOOK_DIR / "outputs" / RUN_TS
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Directorio de salida : {OUT_DIR}\n")


def save_parquet(df: pd.DataFrame, filename: str, description: str = "") -> pathlib.Path:
    path = OUT_DIR / filename
    df.to_parquet(path, index=False, engine="pyarrow")
    print(f"  ✓ {description or filename:<55}  {len(df):>8,} filas")
    return path


def save_csv(df: pd.DataFrame, filename: str, description: str = "") -> pathlib.Path:
    path = OUT_DIR / filename
    df.to_csv(path, index=False)
    print(f"  ✓ {description or filename:<55}  {len(df):>8,} filas")
    return path


def save_plotly_html(fig, filename: str, description: str = "") -> pathlib.Path:
    path = OUT_DIR / filename
    fig.write_html(str(path), include_plotlyjs="cdn")
    print(f"  ✓ {description or filename:<55}")
    return path


# ── Tablas revenue-ranked (matching estructura BQML) ─────────────────────────
save_parquet(df_enriched,    "forecast_h4_enriched.parquet",
             "forecast_h4 enriquecido (todas particiones)")
save_parquet(alerts_margin,  "alerts_top100_h4.parquet",
             "alerts_top100_h4 revenue-rank (VAL)")
save_csv(eval_final,         "run_summary_h4.csv",
         "run_summary_h4 (precision/recall/lift por season)")
save_csv(top20,              "top20_riesgo_economico.csv",
         "top20_riesgo_economico (matching query 7)")

# ── Artefactos margin-ranking (eur_at_risk_margin) ────────────────────────────
save_csv(alerts_top100_h4_margin, "alerts_top100_h4_margin.csv",
         "alerts_top100_h4_margin (VAL, margin-ranked)")
save_csv(eval_margin_alerts,      "eval_alerts_top100_h4_margin_pooled.csv",
         "eval_margin-ranked pooled (por season)")

# ── KPI snapshot (copia en directorio del run para trazabilidad) ──────────────
import shutil as _sh
_kpi_src = NOTEBOOK_DIR / "outputs" / "kpi_por_articulo_snapshot.parquet"
if _kpi_src.exists():
    _sh.copy2(str(_kpi_src), str(OUT_DIR / "kpi_por_articulo_snapshot.parquet"))
    print(f"  ✓ {'kpi_por_articulo_snapshot.parquet (copia run)':<55}")

# ── Gráficos ─────────────────────────────────────────────────────────────────
save_plotly_html(fig1, "fig1_lift_bqml_vs_policyb.html", "Fig1 Lift BQML vs Policy B")
save_plotly_html(fig2, "fig2_riesgo_semanal.html",        "Fig2 Riesgo semanal")
save_plotly_html(fig3, "fig3_scatter_poos_riesgo.html",   "Fig3 Scatter p_oos vs €")
save_plotly_html(fig4, "fig4_top20_skus.html",            "Fig4 Top-20 SKUs")

print(f"\n✅ Exportación completada en {OUT_DIR}")
